**Autor:** Andrés Alejandro Rodríguez Lozano  
**Portafolio:** Portafolio de Andrés Alejandro Rodríguez Lozano

# Capítulo 5 — Drawdowns

$$DD_t = V_t/\max_{s\le t}V_s - 1$$


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%config InlineBackend.figure_formats = ['svg']
plt.rcParams.update({
    'figure.dpi': 72,
    'savefig.dpi': 72,
    'figure.figsize': (8, 3.8),
    'font.size': 9,
    'svg.fonttype': 'none',
    'path.simplify': True,
    'path.simplify_threshold': 0.2,
})

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import portfolio_utils as pu

def load_monthly():
    """Carga precios mensuales: cache mensual, diario, o yfinance."""
    mpath = ROOT / 'data' / 'precios_mensuales.csv'
    if mpath.exists():
        print('Precios mensuales (cache):', mpath.name)
        return pd.read_csv(mpath, index_col=0, parse_dates=True).loc[: '2026-09-30']
    csv = ROOT / 'data' / 'precios_ajustados_diarios.csv'
    if csv.exists():
        daily = pd.read_csv(csv, index_col=0, parse_dates=True)
        print('Precios diarios (cache):', csv.name)
        return pu.to_month_end(daily).loc[: '2026-09-30']
    tickers = sorted(set(list(pu.CURRENT_WEIGHTS) + [pu.BENCHMARK]))
    print('Descargando precios con yfinance…')
    daily = pu.download_prices(tickers)
    return pu.to_month_end(daily).loc[: '2026-09-30']


monthly = load_monthly()
bt = pu.backtest_portfolio(monthly, pu.CURRENT_WEIGHTS, start='2016-01-31', end='2026-09-30', rebalance='A')
spy = pu.backtest_portfolio(monthly, {'SPY': 1}, start='2016-01-31', end='2026-09-30', rebalance='N')
dd_p = bt['value'] / bt['value'].cummax() - 1
dd_s = spy['value'] / spy['value'].cummax() - 1
print('Max DD portafolio:', f'{100*dd_p.min():.2f}%')
print('Max DD SPY       :', f'{100*dd_s.min():.2f}%')


In [ ]:
# Underwater chart
fig, ax = plt.subplots()
ax.fill_between(dd_p.index, dd_p * 100, 0, alpha=0.45, label='Portafolio')
ax.plot(dd_s.index, dd_s * 100, label='SPY', lw=1.4)
ax.set_ylabel('Drawdown (%)')
ax.set_title('Drawdowns — portafolio vs SPY')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## Conclusiones
1. El portafolio asume peores caídas a cambio de mayor CAGR.

## Ejercicios
1. Duración del drawdown 2022.
2. Underwater plot solo post-COVID.
